In [2]:
import xarray as xr
from scipy.interpolate import RegularGridInterpolator
import pandas as pd

from scipy.spatial import cKDTree
import numpy as np

mega

In [3]:
all = pd.read_csv('..\\data\\segment2_FA.csv', sep=',', header=0, names=['lat', 'lon', 'age','q_ihfc','q_ps','q_stein','FA','ps_z','stein_z'])
print(all)

        lat     lon       age   q_ihfc        q_ps     q_stein         FA  \
0     48.75 -127.75  2.906279    200.0  277.518538  299.158617 -25.614542   
1     47.90 -127.20  4.359262    161.0  226.597009  244.266377 -11.433391   
2     47.90 -127.65  3.190760    304.0  264.858297  285.511167   3.032177   
3     48.68 -126.87  5.543758     60.8  200.936360  216.604786 -33.325348   
4     48.69 -126.87  5.543758     63.7  200.936360  216.604786 -33.325348   
...     ...     ...       ...      ...         ...         ...        ...   
3670  47.71 -127.79  3.251128    910.0  262.387792  282.848019   1.003652   
3671  47.71 -127.79  3.251128   1830.0  262.387792  282.848019   1.003652   
3672  47.71 -127.79  3.251128   1330.0  262.387792  282.848019   1.003652   
3673  47.71 -127.79  3.251128  18000.0  262.387792  282.848019   1.003652   
3674  47.71 -127.79  3.251128  18700.0  262.387792  282.848019   1.003652   

             ps_z      stein_z  
0     3096.673436  3222.245155  
1     323

In [4]:
print(all.groupby(['lat', 'lon'])['age'].nunique().max())
print(all.groupby(['lat', 'lon'])[['age', 'q_ps', 'q_stein']].nunique().max())

1
age        1
q_ps       1
q_stein    1
dtype: int64


In [5]:
all = all.drop_duplicates(subset=['lat', 'lon'])

In [19]:
print(all)

        lat     lon       age   q_ihfc         q_ps      q_stein         FA  \
0     48.75 -127.75  2.906279    200.0   277.518538   299.158617 -25.614542   
1     47.90 -127.20  4.359262    161.0   226.597009   244.266377 -11.433391   
2     47.90 -127.65  3.190760    304.0   264.858297   285.511167   3.032177   
3     48.68 -126.87  5.543758     60.8   200.936360   216.604786 -33.325348   
4     48.69 -126.87  5.543758     63.7   200.936360   216.604786 -33.325348   
...     ...     ...       ...      ...          ...          ...        ...   
3075  48.17 -126.70  5.596594    120.0   199.985610   215.579899 -42.689182   
3076  48.70 -126.90  5.543758     75.0   200.936360   216.604786 -27.447968   
3096  47.96 -129.10  0.029513   1471.0  2753.934535  2968.678241   2.136908   
3099  47.96 -129.09  0.029513     39.0  2753.934535  2968.678241   6.714705   
3220  48.49 -128.71  0.591594  10450.0   615.104395   663.068424 -10.749242   

             ps_z      stein_z       z  
0     3096

bath

In [14]:
bathy = pd.read_csv("..\\created_data\\gebco_bath.csv", sep=',', header=0, names=['lon', 'lat', 'z'])
print(bathy)

                 lon        lat       z
0        -132.997917  38.002083 -5025.0
1        -132.993750  38.002083 -5030.0
2        -132.989583  38.002083 -5029.0
3        -132.985417  38.002083 -5028.0
4        -132.981250  38.002083 -5026.0
...              ...        ...     ...
11059195 -121.018750  53.997917   759.0
11059196 -121.014583  53.997917   760.0
11059197 -121.010417  53.997917   760.0
11059198 -121.006250  53.997917   758.0
11059199 -121.002083  53.997917   758.0

[11059200 rows x 3 columns]


interpolate bathy at each all point

In [15]:
bathy_ds = bathy.set_index(['lat', 'lon'])['z'].to_xarray()

interp = bathy_ds.interp(lon=xr.DataArray(all['lon'].values), lat=xr.DataArray(all['lat'].values), method='nearest')

all['z'] = interp.values

print(all)

        lat     lon       age   q_ihfc         q_ps      q_stein         FA  \
0     48.75 -127.75  2.906279    200.0   277.518538   299.158617 -25.614542   
1     47.90 -127.20  4.359262    161.0   226.597009   244.266377 -11.433391   
2     47.90 -127.65  3.190760    304.0   264.858297   285.511167   3.032177   
3     48.68 -126.87  5.543758     60.8   200.936360   216.604786 -33.325348   
4     48.69 -126.87  5.543758     63.7   200.936360   216.604786 -33.325348   
...     ...     ...       ...      ...          ...          ...        ...   
3075  48.17 -126.70  5.596594    120.0   199.985610   215.579899 -42.689182   
3076  48.70 -126.90  5.543758     75.0   200.936360   216.604786 -27.447968   
3096  47.96 -129.10  0.029513   1471.0  2753.934535  2968.678241   2.136908   
3099  47.96 -129.09  0.029513     39.0  2753.934535  2968.678241   6.714705   
3220  48.49 -128.71  0.591594  10450.0   615.104395   663.068424 -10.749242   

             ps_z      stein_z       z  
0     3096

but interp is not good because fills with NaN...

In [20]:
tree = cKDTree(bathy[['lat', 'lon']].values)
dist, idx = tree.query(all[['lat', 'lon']].values, k=1)
all['z'] = bathy['z'].values[idx]

print(all[244:255])

       lat     lon       age  q_ihfc        q_ps     q_stein        FA  \
472  47.78 -127.35  4.265892   154.0  229.063401  246.925091 -4.742085   
474  47.78 -127.36  4.110389   158.0  233.356115  251.552538 -4.742085   
475  47.78 -127.37  4.110389   181.0  233.356115  251.552538 -4.742085   
476  47.79 -127.39  4.110389   155.0  233.356115  251.552538 -4.742085   
477  47.79 -127.40  4.110389   145.0  233.356115  251.552538 -4.742085   
478  47.84 -127.72  3.371136  1164.0  257.675152  277.767902 -0.975070   
479  47.84 -127.74  3.371136   759.0  257.675152  277.767902 -0.975070   
480  47.84 -127.75  3.161246   310.0  266.091792  286.840846 -0.975070   
481  47.84 -127.77  3.161246   253.0  266.091792  286.840846 -0.975070   
483  47.85 -127.78  2.964588   194.0  274.775784  296.201991 -0.518283   
484  47.85 -127.79  2.964588   213.0  274.775784  296.201991 -0.518283   

            ps_z      stein_z       z  
472  3222.891300  3353.872355 -2620.0  
474  3209.593318  3340.004460 -

In [21]:
all.to_csv('..\\created_data\\kd_all_bathy.csv', index=False)